# Blitz ASR - AWS Batch Deployment

Deploy and test AWS Batch for production-scale audio processing.

## Overview

AWS Batch provides event-driven processing triggered by S3 uploads:

```
S3 Input --> EventBridge --> AWS Batch --> S3 Output
```

## Prerequisites

- AWS CLI configured with appropriate permissions
- Blitz image in ECR (see notebook 01)
- VPC with private subnets (NAT gateway for ECR access)

## This Notebook Can Run From

- Mac Silicon (development machine with AWS CLI)
- g6.xlarge instance
- Any machine with AWS CLI access

---
## 1. Configuration

Set your AWS account details.

In [ ]:
# AWS Configuration - UPDATE THESE VALUES
ACCOUNT_ID = "123456789012"  # Your AWS account ID
REGION = "us-east-1"         # Your AWS region
ENV = "dev"                  # Environment: dev, staging, prod

# ECR image (must exist - see notebook 01)
ECR_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/blitz-asr:latest"

print(f"Account: {ACCOUNT_ID}")
print(f"Region: {REGION}")
print(f"Environment: {ENV}")
print(f"ECR URI: {ECR_URI}")

---
## 2. Verify ECR Image Exists

Ensure the Blitz image is available in ECR.

In [ ]:
# Check if image exists in ECR
!aws ecr describe-images \
    --repository-name blitz-asr \
    --region {REGION} \
    --query 'imageDetails[*].[imagePushedAt,imageTags]' \
    --output table

In [ ]:
# If image doesn't exist, push it first
# (Run this cell only if needed)
# !./scripts/push_to_ecr.sh {ACCOUNT_ID} {REGION} latest

---
## 3. Infrastructure Discovery

Find your VPC and subnets for the Batch compute environment.

In [ ]:
# List available VPCs
!aws ec2 describe-vpcs \
    --region {REGION} \
    --query 'Vpcs[*].[VpcId,Tags[?Key==`Name`].Value|[0],CidrBlock]' \
    --output table

In [ ]:
# Set your VPC ID - UPDATE THIS
VPC_ID = "vpc-xxxxxxxxx"  # Replace with your VPC ID

In [ ]:
# List subnets in the VPC
!aws ec2 describe-subnets \
    --region {REGION} \
    --filters Name=vpc-id,Values={VPC_ID} \
    --query 'Subnets[*].[SubnetId,AvailabilityZone,CidrBlock,Tags[?Key==`Name`].Value|[0]]' \
    --output table

In [ ]:
# Set your subnet IDs (comma-separated) - UPDATE THESE
# Use private subnets with NAT gateway access for ECR
SUBNET_IDS = "subnet-aaa,subnet-bbb"  # Replace with your subnet IDs

---
## 4. Deploy CloudFormation Stack

Deploy the AWS Batch infrastructure using CloudFormation.

In [ ]:
# Navigate to project root
import os
os.chdir('..')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Deploy using the provided script
!./scripts/deploy_batch.sh \
    --environment {ENV} \
    --region {REGION} \
    --vpc-id {VPC_ID} \
    --subnet-ids {SUBNET_IDS} \
    --ecr-image {ECR_URI}

In [ ]:
# Check stack status
!aws cloudformation describe-stacks \
    --stack-name blitz-batch-{ENV} \
    --region {REGION} \
    --query 'Stacks[0].[StackStatus,CreationTime]' \
    --output table

---
## 5. Get Stack Outputs

Retrieve the S3 bucket names and other resources created by the stack.

In [ ]:
import boto3

cf = boto3.client('cloudformation', region_name=REGION)
response = cf.describe_stacks(StackName=f'blitz-batch-{ENV}')
outputs = response['Stacks'][0]['Outputs']

# Extract bucket names
stack_outputs = {o['OutputKey']: o['OutputValue'] for o in outputs}

INPUT_BUCKET = stack_outputs.get('InputBucket', 'NOT_FOUND')
OUTPUT_BUCKET = stack_outputs.get('OutputBucket', 'NOT_FOUND')

print(f"Input Bucket: {INPUT_BUCKET}")
print(f"Output Bucket: {OUTPUT_BUCKET}")
print(f"\nAll outputs:")
for k, v in stack_outputs.items():
    print(f"  {k}: {v}")

---
## 6. Upload Test Files

Upload audio files to S3 to trigger transcription jobs.

In [ ]:
from pathlib import Path
import boto3
from tqdm import tqdm

s3 = boto3.client('s3', region_name=REGION)
fixtures = Path("tests/fixtures")

# Select files to upload (limited set for testing)
test_files = [
    fixtures / "english.mp3",
    fixtures / "french.mp3",
    fixtures / "german.mp3",
    fixtures / "spanish.mp3",
]

# Upload files
for filepath in tqdm(test_files, desc="Uploading to S3"):
    if filepath.exists():
        s3.upload_file(str(filepath), INPUT_BUCKET, filepath.name)
        print(f"  Uploaded: {filepath.name}")
    else:
        print(f"  Skipped (not found): {filepath}")

In [ ]:
# Verify uploads
!aws s3 ls s3://{INPUT_BUCKET}/

---
## 7. Monitor Jobs

Watch the AWS Batch jobs and CloudWatch logs.

In [ ]:
# List recent Batch jobs
!aws batch list-jobs \
    --job-queue blitz-job-queue-{ENV} \
    --region {REGION} \
    --query 'jobSummaryList[*].[jobId,jobName,status,createdAt]' \
    --output table

In [ ]:
# Watch CloudWatch logs (press Ctrl+C to stop)
!aws logs tail /aws/batch/blitz-{ENV} --follow --region {REGION}

In [ ]:
# Check job status programmatically
import boto3
import time

batch = boto3.client('batch', region_name=REGION)

def get_job_status():
    response = batch.list_jobs(
        jobQueue=f'blitz-job-queue-{ENV}',
        jobStatus='RUNNING'
    )
    running = len(response.get('jobSummaryList', []))
    
    response = batch.list_jobs(
        jobQueue=f'blitz-job-queue-{ENV}',
        jobStatus='SUCCEEDED'
    )
    succeeded = len(response.get('jobSummaryList', []))
    
    response = batch.list_jobs(
        jobQueue=f'blitz-job-queue-{ENV}',
        jobStatus='FAILED'
    )
    failed = len(response.get('jobSummaryList', []))
    
    return running, succeeded, failed

# Poll until jobs complete
print("Waiting for jobs to complete...")
for _ in range(60):  # Wait up to 10 minutes
    running, succeeded, failed = get_job_status()
    print(f"Running: {running}, Succeeded: {succeeded}, Failed: {failed}")
    if running == 0:
        break
    time.sleep(10)

print(f"\nFinal: {succeeded} succeeded, {failed} failed")

---
## 8. Download Results

Retrieve transcription results from the output bucket.

In [ ]:
# List output files
!aws s3 ls s3://{OUTPUT_BUCKET}/transcripts/ --recursive

In [ ]:
# Download results
!mkdir -p ./batch-results
!aws s3 sync s3://{OUTPUT_BUCKET}/transcripts/ ./batch-results/

In [ ]:
# Analyze results
import json
from pathlib import Path

results_dir = Path("./batch-results")
results = []

for result_file in results_dir.glob("*.json"):
    with open(result_file) as f:
        data = json.load(f)
    
    results.append({
        "file": result_file.name,
        "language": data.get("transcription", {}).get("language", {}).get("code", "unknown"),
        "duration_sec": data.get("audio", {}).get("duration_sec", 0),
        "rtf": data.get("metrics", {}).get("rtf", 0),
        "text_preview": data.get("transcription", {}).get("text", "")[:100]
    })

for r in results:
    print(f"{r['file']}:")
    print(f"  Language: {r['language']}")
    print(f"  Duration: {r['duration_sec']:.1f}s")
    print(f"  RTF: {r['rtf']:.4f} ({1/r['rtf']:.0f}x real-time)")
    print(f"  Preview: {r['text_preview']}...")
    print()

---
## 9. Cost Analysis

Calculate the actual cost per 1000 minutes of audio.

In [ ]:
# g6.xlarge pricing (as of 2024)
ON_DEMAND_HOURLY = 0.8049  # $/hour
SPOT_HOURLY = 0.24        # $/hour (approximate, varies)

# Calculate from results
total_audio_sec = sum(r['duration_sec'] for r in results)
avg_rtf = sum(r['rtf'] for r in results) / len(results) if results else 0.01

# Time to process 1000 minutes of audio
audio_1000_min_sec = 1000 * 60  # 60,000 seconds
processing_time_sec = audio_1000_min_sec * avg_rtf
processing_time_hours = processing_time_sec / 3600

cost_on_demand = processing_time_hours * ON_DEMAND_HOURLY
cost_spot = processing_time_hours * SPOT_HOURLY

print(f"Average RTF: {avg_rtf:.4f}")
print(f"Processing time for 1000 min audio: {processing_time_hours:.2f} hours")
print(f"")
print(f"Cost per 1000 minutes:")
print(f"  On-demand: ${cost_on_demand:.2f}")
print(f"  Spot:      ${cost_spot:.2f}")

---
## 10. Scale Testing

Process a larger batch of files to test at scale.

In [ ]:
# Upload all fixtures for scale testing
from pathlib import Path
from tqdm import tqdm
import boto3

s3 = boto3.client('s3', region_name=REGION)
fixtures = Path("tests/fixtures")

# Get all audio files (excluding very large ones)
all_files = list(fixtures.glob("*.mp3")) + list(fixtures.glob("*.wav"))
all_files = [f for f in all_files if f.stat().st_size < 50_000_000]  # <50MB

print(f"Found {len(all_files)} files for scale testing")

# Uncomment to upload
# for filepath in tqdm(all_files, desc="Uploading"):
#     s3.upload_file(str(filepath), INPUT_BUCKET, filepath.name)

---
## 11. Cleanup

Remove resources when done testing.

In [ ]:
# Clear input bucket (keeps bucket, removes files)
# !aws s3 rm s3://{INPUT_BUCKET}/ --recursive

In [ ]:
# Clear output bucket
# !aws s3 rm s3://{OUTPUT_BUCKET}/ --recursive

In [ ]:
# Delete CloudFormation stack (removes all resources)
# WARNING: This will delete S3 buckets and all their contents!
# !aws cloudformation delete-stack --stack-name blitz-batch-{ENV} --region {REGION}

In [ ]:
# Clean up local results
# !rm -rf ./batch-results

---
## Next Steps

- **Production Deployment**: Update `ENV` to `prod` and redeploy
- **Scaling**: Adjust `MaxvCpus` in CloudFormation for higher throughput
- **Monitoring**: Set up CloudWatch alarms for job failures
- **Cost Optimization**: Use Spot instances for batch processing

## Documentation

- `docs/BATCH_PROCESSING.md` - Full AWS Batch documentation
- `docs/ARCHITECTURE.md` - System architecture
- `infrastructure/batch/cloudformation.yaml` - CloudFormation template